# Logger: PGN + JSONL game records

**Deliverable:** logging. Every game produces

* `<game_id>.jsonl`: one JSON object per ply, written and flushed immediately (a crash mid-game
  keeps everything up to the last move). Fields: FEN before/after, move (UCI + SAN), who moved,
  eval, engine best move, think time, robot attempts, arm plan, free-form extras.
* `<game_id>.pgn`: the game in standard PGN, with per-move comments using the common
  `[%eval ...]` and `[%emt ...]` annotations (readable by lichess / chess GUIs), written at the end.

These files feed the teaching side (`gambit_teach/notebooks/run_analyze_history.ipynb`) and the
README measurement logs (pick-and-place success, retries, game statistics).

**TODO checklist**
- [ ] PGN comment helpers (`format_emt`, `move_comment`).
- [ ] JSONL and PGN writer (`GameLogger`), readers back to pandas / python-chess.
- [ ] Self-checks (plan at the bottom).
- [ ] Decide the player id scheme (name? hashed?) and keep personal data out of shared logs.
- [ ] Log recovery events (kind, action) as separate JSONL lines (`kind: "recovery"`).

In [ ]:
from __future__ import annotations

import datetime as dt
import io
import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional

import chess
import chess.pgn
import pandas as pd

## Per-move record

`eval_cp` is from **White's** point of view (the PGN convention), after the move.

In [ ]:
@dataclass
class MoveRecord:
    game_id: str
    ply: int
    by: str                               # "human" | "robot"
    fen_before: str
    fen_after: str
    uci: str
    san: str
    think_time_s: float = 0.0
    eval_cp: Optional[int] = None         # White POV, after the move
    best_uci: Optional[str] = None        # engine's best move in fen_before
    robot_attempts: int = 0
    plan: str = ""
    timestamp: str = field(default_factory=lambda: dt.datetime.now().isoformat(timespec="seconds"))
    extra: dict = field(default_factory=dict)
    kind: str = "move"


def format_emt(seconds: float) -> str:
    """Seconds -> 'H:MM:SS' as used by the PGN [%emt] annotation (e.g. 3725 -> '1:02:05')."""
    # Design: round to whole seconds, clamp negatives to 0, integer division for h/m/s.
    # TODO: implement
    raise NotImplementedError


def move_comment(r: MoveRecord) -> str:
    """PGN comment for one move: '[%eval 0.20] [%emt 0:00:03]' plus optional notes."""
    # Design:
    # - [%eval] in pawns (eval_cp / 100), only if an eval exists; [%emt] always.
    # - Free-text extras that help later analysis: robot attempts (robot moves with retries),
    #   the engine's best move when it differs from the played one.
    # - python-chess parses [%eval]/[%emt] back (GameNode.eval(), GameNode.emt()), so keep the
    #   exact annotation syntax.
    # TODO: implement
    raise NotImplementedError

## Writer

In [ ]:
class GameLogger:
    def __init__(self, log_dir: Path | str, game_id: Optional[str] = None, headers: Optional[dict] = None,
                 start_fen: str = chess.STARTING_FEN):
        """Create the log folder and open the JSONL file for appending."""
        # Design:
        # - game_id defaults to a timestamp ("game_YYYYmmdd_HHMMSS"); paths <dir>/<id>.jsonl/.pgn
        #   stored as jsonl_path / pgn_path.
        # - PGN headers: sensible defaults (Event, Site, Date in PGN format YYYY.MM.DD) overridden
        #   by `headers`.
        # - Keep records in memory for the PGN; keep the JSONL file handle open.
        # TODO: implement
        raise NotImplementedError

    def log(self, record: MoveRecord) -> None:
        """Append one move: keep it in memory and write one JSON line, flushed immediately."""
        # Design: dataclasses.asdict + json.dumps; flush so a crash keeps everything so far.
        # TODO: implement
        raise NotImplementedError

    def log_event(self, kind: str, **data) -> None:
        """Non-move lines (recovery, errors, notes) in the same JSONL file."""
        # Design: same file, a `kind` field distinguishes them from moves; include game_id + timestamp.
        # TODO: implement
        raise NotImplementedError

    def to_pgn(self, result: str = "*") -> chess.pgn.Game:
        """Build a python-chess Game from the records (mainline + per-move comments)."""
        # Design: chess.pgn.Game, copy headers, Game.setup for non-standard start FENs, Result
        # header, then add_variation per record with move_comment as the node comment.
        # TODO: implement
        raise NotImplementedError

    def finish(self, result: str = "*") -> Path:
        """Write the PGN file, close the JSONL file, return the PGN path."""
        # TODO: implement
        raise NotImplementedError

    def close(self) -> None:
        """Close the JSONL file handle (idempotent)."""
        # TODO: implement
        raise NotImplementedError

## Readers

In [ ]:
def read_jsonl(path: Path | str, kind: Optional[str] = "move") -> pd.DataFrame:
    """JSONL log -> DataFrame, keeping only lines of `kind` (None = keep every line)."""
    # TODO: implement
    raise NotImplementedError


def read_pgn_games(path: Path | str) -> list[chess.pgn.Game]:
    """Every game in a PGN file (chess.pgn.read_game until it returns None)."""
    # TODO: implement
    raise NotImplementedError

## Self-check (writes only to a temporary folder)

In [ ]:
import tempfile

# Self-check plan (write these as asserts once the functions above work; use tempfile.TemporaryDirectory):
# - Log three moves (e4 e5 Nf3) with think times and evals, plus one log_event("recovery", ...).
# - read_jsonl returns 3 move rows with SAN e4/e5/Nf3; kind=None returns 4 rows.
# - The PGN reads back with one game whose mainline is e2e4 e7e5 g1f3.
# - First node: eval() is +20 cp for White, emt() matches the logged think time (rounded to s).
# - A robot move with attempts shows "robot attempts=..." in its comment.
# - format_emt(3725) == "1:02:05".
# Example shape of a test:
# assert format_emt(3725) == "1:02:05"
print("Self-check: not written yet (see plan above)")